### 1. load libraries

In [12]:
import pandas as pd
import glob
import re
import numpy as np
from datetime import datetime
import gc
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
gc.collect()

8

### 2. Load ICD-10 data

In [13]:
icd_path = "/mnt/d/pydatascience/g3_regress/data/icd10"

# Load aa Ca csv files
icd_files = glob.glob(f"{icd_path}/icd*.csv")

# Function to simplify column names
def simplify_column_names(df):
    simplified_names = {}
    for col in df.columns:
        # Basic simplifications: lower case, replace spaces and hyphens
        simple_name = col.lower().replace(' ', '_').replace('-', '_')
        
        # Specific simplifications for this dataset
        if 'diagnosis_code' in simple_name:
            # Extract the number and create a simplified name
            number = simple_name.split('_')[3]  # Assuming the number is always in the same position
            simple_name = f'diagnosis_code_{number}'
        elif 'diagnosis_description' in simple_name:
            number = simple_name.split('_')[3]
            simple_name = f'diagnosis_desc_{number}'
        elif 'diagnosis_comment' in simple_name:
            number = simple_name.split('_')[3]
            simple_name = f'diagnosis_comment_{number}'
        elif 'diagnosis_term_id' in simple_name:
            number = simple_name.split('_')[3]
            simple_name = f'diagnosis_term_id_{number}'
        elif 'diagnosis_status' in simple_name:
            number = simple_name.split('_')[3]
            simple_name = f'diagnosis_status_{number}'
        
        simplified_names[col] = simple_name

    df.rename(columns=simplified_names, inplace=True)
    columns_to_drop = [f'diagnosis_status_{i}' for i in range(1, 21)]
    df = df.drop(columns=columns_to_drop)
    columns_to_drop = [f'diagnosis_term_id_{i}' for i in range(1, 21)]
    df = df.drop(columns=columns_to_drop)
    df = df.drop(columns=['diagnosis_comment_comment'])
    return df

# List to store all dataframes
icd_dataframes = []

for file in icd_files:
    df = pd.read_csv(file, skiprows=1)  # Skip the first row if it's not part of the data
    df = simplify_column_names(df)
    icd_dataframes.append(df)

# Concatenate all dataframes into one
icd_df = pd.concat(icd_dataframes, ignore_index=True)
icd_df = icd_df.drop_duplicates()

# Drop unnecessary 'status' and 'term_id' columns
columns_to_drop = [col for col in icd_df.columns if 'status' in col or 'term_id' in col]
icd_df.drop(columns=columns_to_drop, inplace=True)
icd_df = icd_df.rename(columns={'reference_key': 'key',
                                'appointment_date_(yyyy_mm_dd)':'date'})
icd_df['date'] = pd.to_datetime(icd_df['date'], format='%Y-%m-%d')
# Create a list to store code and description pairs
code_desc_pairs = []

# Iterate over each pair of diagnosis code and description columns
for i in range(1, 21):  # Adjust the range according to how many diagnosis pairs you have
    code_column = f'diagnosis_code_{i}'
    desc_column = f'diagnosis_desc_{i}'

    # Drop rows where either code or description is NaN
    subset = icd_df[[code_column, desc_column]].dropna()

    # Append to the list
    code_desc_pairs.append(subset)

all_pairs = pd.concat(code_desc_pairs)[['diagnosis_code_1', 'diagnosis_desc_1']]
all_pairs.columns = ['diagnosis_code', 'diagnosis_desc']
# Group by both 'diagnosis_code' and 'diagnosis_desc' to ensure descriptions are correctly matched
icd_df_map = all_pairs.groupby(['diagnosis_code', 'diagnosis_desc']).size().reset_index(name='Frequency')

# lasheen WP, Cordier T, Gumpina R, Haugh G, Davis J, Renda A. Charlson Comorbidity Index: ICD-9 Update and ICD-10 Translation. American Health & Drug Benefits. 2019;12(4): 188–197. 

# Hypertension: I10, I12.9, I12, I11, I11.9, I13.0, I 13.1, I13.2, I13.9, I15.0, I15.1, I15.9, I67.4
hypertension_related_codes = icd_df_map[icd_df_map['diagnosis_code'].str.startswith('I1')]
# Diabetes:
diabetes_related_codes = icd_df_map[icd_df_map['diagnosis_code'].str.startswith('E1')]
# Drop hypoglycemia related codes:
diabetes_related_codes = diabetes_related_codes[~diabetes_related_codes['diagnosis_code'].str.startswith('E16')]
# cardiovascular disease related codes:
# The SPRINT Research Group. A Randomized Trial of Intensive versus Standard Blood-Pressure Control. New England Journal of Medicine. 2015;373(22): 2103–2116. https://doi.org/10.1056/NEJMoa1511939.
chf_related_codes = icd_df_map[icd_df_map['diagnosis_code'].str.contains(r'^I2[0-1]', case=False, na=False)]
ihd_related_codes = icd_df_map[icd_df_map['diagnosis_code'].str.contains(r'^I2[0-5]', case=False, na=False)]
stroke_related_codes = icd_df_map[icd_df_map['diagnosis_code'].str.startswith('I6')]
pvd_related_codes = icd_df_map[icd_df_map['diagnosis_code'].str.contains(r'^I7[0-4]', case=False, na=False)]

# Combine the three dataframes
sprint_related_codes = pd.concat([chf_related_codes, ihd_related_codes, stroke_related_codes]).drop_duplicates().reset_index(drop=True)

gc.collect()

/tmp/ipykernel_3893034/2913391623.py:45: DtypeWarning: Columns (51) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, skiprows=1)  # Skip the first row if it's not part of the data


44

In [18]:
def match_code(code, exact=None, prefix=None, ranges=None):
    if exact and code in exact:
        return True
    if prefix and any(code.startswith(p) for p in prefix):
        return True
    if ranges:
        root = re.match(r"[A-Z]+\d+", code)
        if root:
            root = root.group(0)
            for start, end in ranges:
                if start <= root <= end:
                    return True
    return False

# Dictionary of CCI conditions
cci_conditions = {
    "myocardial_infarction": {"prefix": ["I21", "I22", "I25.2"]},
    "congestive_heart_failure": {"exact": ["I11.0", "I13.0", "I13.2", "I25.5", "I42.0", "I42.5", "I42.6", "I42.7", "I42.8", "I42.9", "P29.0"], "prefix": ["I43", "I50"]},
    "peripheral_vascular_disease": {"exact": ["I73.1", "I73.8", "I73.9", "I77.1", "I79.0", "I79.1", "I79.8", "K55.1", "K55.8", "K55.9", "Z95.8", "Z95.9"], "prefix": ["I70", "I71"]},
    "cerebrovascular_disease": {"prefix": ["G45", "G46", "H34.0", "H34.1", "H34.2", "I60", "I61", "I62", "I63", "I64", "I65", "I66", "I67", "I68"]},
    "dementia": {"exact": ["F04", "F05", "F06.1", "F06.8", "G13.2", "G13.8", "G31.1", "G31.2", "G91.4", "G94", "R41.81", "R54"], "prefix": ["F01", "F02", "F03", "G30", "G31.0"]},
    "chronic_pulmonary_disease": {"exact": ["J68.4", "J70.1", "J70.3"], "prefix": ["J40", "J41", "J42", "J43", "J44", "J45", "J46", "J47", "J60", "J61", "J62", "J63", "J64", "J65", "J66", "J67"]},
    "rheumatic_disease": {"exact": ["M31.5", "M35.1", "M35.3", "M36.0"], "prefix": ["M05", "M06", "M32", "M33", "M34"]},
    "peptic_ulcer_disease": {"prefix": ["K25", "K26", "K27", "K28"]},
    "mild_liver_disease": {"exact": ["K70.0", "K70.1", "K70.2", "K70.3", "K70.9", "K71.3", "K71.4", "K71.5", "K71.7", "K76.0", "K76.2", "K76.3", "K76.4", "K76.8", "K76.9", "Z94.4"], "prefix": ["B18", "K73", "K74"]},
    "diabetes_wo_complication": {"prefix": ["E08", "E09", "E10", "E11", "E13"], "subcode_in": [".0", ".1", ".6", ".8", ".9"]},
    "renal_mild_moderate": {"exact": ["I12.9", "I13.0", "I13.10", "N18.1", "N18.2", "N18.3", "N18.4", "N18.9", "Z94.0"], "prefix": ["N03", "N05"]},
    "diabetes_w_complication": {"prefix": ["E08", "E09", "E10", "E11", "E13"], "subcode_in": [".2", ".3", ".4", ".5"]},
    "hemiplegia_paraplegia": {"exact": ["G04.1", "G11.4", "G80.0", "G80.1", "G80.2"], "prefix": ["G81", "G82", "G83"]},
    "any_malignancy": {"exact": ["C43", "C50", "C76", "C80.1"], "prefix": ["C0", "C1", "C2", "C30", "C31", "C32", "C33", "C34", "C37", "C38", "C39", "C40", "C41", "C45", "C46", "C47", "C48", "C49", "C51", "C52", "C53", "C54", "C55", "C56", "C57", "C58", "C60", "C61", "C62", "C63", "C81", "C82", "C83", "C84", "C85", "C88", "C90", "C91", "C92", "C93", "C94", "C95", "C96"]},
    "liver_severe": {"exact": ["I86.4", "K76.5", "K76.6", "K76.7"], "prefix": ["I85.0", "K70.4", "K71.1", "K72.1", "K72.9"]},
    "renal_severe": {"exact": ["I12.0", "I13.11", "I13.2", "N18.5", "N18.6", "N25.0", "Z99.2"], "prefix": ["N19", "Z49"]},
    "hiv": {"prefix": ["B20"]},
    "metastatic_cancer": {"exact": ["C80.0", "C80.2"], "prefix": ["C77", "C78", "C79"]},
    "aids": {"exact": ["A07.2", "A07.3", "A02.1", "A81.2", "B59", "Z87.01", "R64", "B00", "B58"], "prefix": ["B37", "C53", "B38", "B45", "B25", "G93.4", "B39", "C46", "A31", "B58"], "ranges": [("C81", "C96"), ("A15", "A19")]}
}

df_icd = pd.read_csv("/mnt/d/pydatascience/g3_regress/data/icd_df_map.csv")
icd_col = 'diagnosis_code'
df_condition_map = {}
for condition, rules in cci_conditions.items():
    if condition in ["diabetes_wo_complication", "diabetes_w_complication"]:
        # Special case: needs subcode match
        def subcode_match(code, main, subs):
            if any(code.startswith(m) for m in main):
                try:
                    return f".{code.split('.')[1][:1]}" in subs
                except IndexError:
                    return False
            return False
        func = lambda x: subcode_match(x, rules["prefix"], rules["subcode_in"])
    else:
        func = lambda x: match_code(x, rules.get("exact"), rules.get("prefix"), rules.get("ranges"))
    df_condition_map[condition] = df_icd[df_icd[icd_col].astype(str).apply(func)].drop_duplicates().reset_index(drop=True)


In [19]:
df_condition_map

{'myocardial_infarction':    diagnosis_code                      diagnosis_desc  Frequency
 0             I21            Acute myocardial infarct          3
 1           I21.0               Anterior AMI, initial        122
 2           I21.0          Anterolateral AMI, initial         45
 3           I21.0           Anteroseptal AMI, initial         85
 4           I21.1               Inferior AMI, initial        158
 5           I21.2            AMI initial care episode         40
 6           I21.2                Lateral AMI, initial          8
 7           I21.2              Posterior AMI, initial         17
 8           I21.4                  Subendocardial AMI        146
 9           I21.4  Subendocardial AMI initial episode       1540
 10          I21.9         AMI initial episode of care         27
 11          I21.9         Acute myocardial infarction         10
 12          I21.9          Coronary artery thrombosis          6
 13          I25.2                              Old

In [20]:
# Flatten all diagnosis code columns into one per-row set
diagnosis_code_cols = [col for col in icd_df.columns if col.startswith("diagnosis_code_")]
cci_df = icd_df[["key", "date"]].copy()
cci_conditions_list = list(df_condition_map.keys())
# Create a set for fast lookup for each condition
condition_code_sets = {
    cond: set(df_condition_map[cond]["diagnosis_code"].dropna().astype(str).str.upper())
    for cond in cci_conditions_list
}
# Apply row-wise check
for idx, row in icd_df.iterrows():
    diag_codes = {str(row[col]).upper() for col in diagnosis_code_cols if pd.notna(row[col])}
    for condition, code_set in condition_code_sets.items():
        if diag_codes & code_set:  # intersection exists
            cci_df.at[idx, condition] = 1

In [30]:
cci_df

,key,date,congestive_heart_failure,cerebrovascular_disease,renal_mild_moderate,hemiplegia_paraplegia,dementia,aids,diabetes_w_complication,diabetes_wo_complication,peptic_ulcer_disease,any_malignancy,chronic_pulmonary_disease,myocardial_infarction,renal_severe,rheumatic_disease,mild_liver_disease,metastatic_cancer,peripheral_vascular_disease,liver_severe
0,2120516,2010-10-06,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2120516,2010-11-23,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2120516,2011-01-26,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2120516,2011-05-18,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2120516,2011-09-07,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1116616,1680317,2022-06-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1116617,1680317,2022-09-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1116618,1680317,2023-02-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1116619,1680317,2023-04-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
# Step 1–3: group, sort, and forward-fill within each group
cci_df_sorted = (
    cci_df
    .sort_values(["key", "date"])  # sort by patient and time
    .groupby("key", group_keys=False)  # group by patient
    .apply(lambda group: group.ffill())  # forward fill within group
    .reset_index(drop=True)  # reset index after apply
)

/tmp/ipykernel_3893034/4054395794.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.ffill())  # forward fill within group


In [31]:
# Load demographic data
demo_df = pd.read_csv("/mnt/d/pydatascience/g3_regress/data/demographic_df.csv")
demo_df.columns = ['key', 'dob', 'gender']
demo_df['key'] = demo_df['key'].astype(int)
demo_df['dob'] = pd.to_datetime(demo_df['dob'])

# Merge with CCI dataframe
merged_df = pd.merge(cci_df_sorted, demo_df, on="key", how="left")

# Calculate age at each diagnosis date
merged_df["age"] = (merged_df["date"] - merged_df["dob"]).dt.days // 365

# Define Charlson Comorbidity Index weights
cci_weights = {
    'myocardial_infarction': 1,
    'congestive_heart_failure': 1,
    'peripheral_vascular_disease': 1,
    'cerebrovascular_disease': 1,
    'dementia': 1,
    'chronic_pulmonary_disease': 1,
    'rheumatic_disease': 1,
    'peptic_ulcer_disease': 1,
    'mild_liver_disease': 1,
    'diabetes_wo_complication': 1,
    'diabetes_w_complication': 2,
    'hemiplegia_paraplegia': 2,
    'renal_mild_moderate': 1,
    'renal_severe': 3,
    'any_malignancy': 2,
    'metastatic_cancer': 6,
    'liver_severe': 3,
    'hiv': 3,
    'aids': 6
}

# Fill NaNs with 0 for only those columns that exist in the DataFrame
for col in cci_weights.keys():
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].fillna(0)

# Safely compute CCI score only for existing columns
merged_df["cci_score"] = sum(
    merged_df[col] * weight for col, weight in cci_weights.items() if col in merged_df.columns
)

# Age adjustment for CCI: 1 point per decade above 50
merged_df["age_points"] = pd.cut(
    merged_df["age"],
    bins=[0, 49, 59, 69, 79, 89, 200],
    labels=[0, 1, 2, 3, 4, 5],
    right=True
).astype(int)

# Total CCI Score = CCI + age adjustment
merged_df["cci_score_total"] = merged_df["cci_score"] + merged_df["age_points"]

In [26]:
# Extract relevant columns
cci_score_export = merged_df[["key", "date", "cci_score_total"]]

# Save to CSV
output_path = "/mnt/d/pydatascience/g3_regress/data/cci/cci_output.csv"
cci_score_export.to_csv(output_path, index=False)


In [11]:
cci_score_export.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1116621 entries, 0 to 1116620
Data columns (total 3 columns):
 #   Column           Non-Null Count    Dtype         
---  ------           --------------    -----         
 0   key              1116621 non-null  int64         
 1   date             1116621 non-null  datetime64[ns]
 2   cci_score_total  1116621 non-null  int64         
dtypes: datetime64[ns](1), int64(2)
memory usage: 25.6 MB


In [34]:
merged_df.columns

Index(['key', 'date', 'congestive_heart_failure', 'cerebrovascular_disease',
       'renal_mild_moderate', 'hemiplegia_paraplegia', 'dementia', 'aids',
       'diabetes_w_complication', 'diabetes_wo_complication',
       'peptic_ulcer_disease', 'any_malignancy', 'chronic_pulmonary_disease',
       'myocardial_infarction', 'renal_severe', 'rheumatic_disease',
       'mild_liver_disease', 'metastatic_cancer',
       'peripheral_vascular_disease', 'liver_severe', 'dob', 'gender', 'age',
       'cci_score', 'age_points', 'cci_score_total'],
      dtype='object')

In [ ]:
merged_df[['key', 'date', 'congestive_heart_failure', 'cerebrovascular_disease',
       'renal_mild_moderate', 'hemiplegia_paraplegia', 'dementia', 'aids',
       'diabetes_w_complication', 'diabetes_wo_complication',
       'peptic_ulcer_disease', 'any_malignancy', 'chronic_pulmonary_disease',
       'myocardial_infarction', 'renal_severe', 'rheumatic_disease',
       'mild_liver_disease', 'metastatic_cancer',
       'peripheral_vascular_disease', 'liver_severe', 'age', 'cci_score_total']]

,key,date,congestive_heart_failure,cerebrovascular_disease,renal_mild_moderate,hemiplegia_paraplegia,dementia,aids,diabetes_w_complication,diabetes_wo_complication,...,any_malignancy,chronic_pulmonary_disease,myocardial_infarction,renal_severe,rheumatic_disease,mild_liver_disease,metastatic_cancer,peripheral_vascular_disease,liver_severe,cci_score_total
0,449,2009-09-10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0
1,449,2009-09-24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0
2,449,2009-12-17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0
3,449,2010-03-11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0
4,449,2010-07-14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1116616,13239674,2023-10-25,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
1116617,13239674,2023-10-31,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
1116618,13241503,2023-11-01,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
1116619,13280731,2023-11-23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0
